# Capstone model experiments (xp) — run xp-05

Rules of the road live in `work/experiments/README.md`. Short version:

- Same label, same cutoffs, same client-holdout split as the main line (w05).
- Every feature must be knowable at the decision moment (2026-03-01).
- No tuning on the holdout: an inner client-holdout inside train decides.
- A win = beat the rule at BOTH precision@10 and precision@50 on seed 42 AND
  confirm on other seeds.

History: xp-01 and xp-02 inflated numbers via leaked July-snapshot dim_content
columns (updated date, word count); xp-03 ran technique-only on the safe w05
features and was a clean negative (rule 30/44 beat every challenger); xp-04
added XGBoost/LightGBM, but their config (scale_pos_weight + AUC early
stopping) degenerated to ~0% — a config artifact, not a claim about the
libraries.

xp-05 exhausts the space in one run:

1. **Fixed, fair XGBoost/LightGBM** (logloss, early stopping on the inner
   split, no scale_pos_weight) plus **CatBoost**, a small neural net (MLP),
   k-nearest neighbors, SGD-logistic, GaussianNB, and a stacking meta-model.
2. **New decision-time-safe features** the rule cannot see: Jan->Feb impression
   trend (momentum), an engagement-gap feature, and page age.
3. **Four client-holdout seeds (42-45)** with a fixed-config zoo, so the rule's
   win is checked on average, not one lucky split.
4. **AUPRC and lift@50** reported alongside precision.

Realistic expectation: the transparent rule still wins. That is the honest
result; xp-05 makes it bulletproof and shows the breadth of what was tried.

Run top to bottom.

In [47]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Same w05 load (same rows, same order => the rule must reproduce 30/44 on seed 42),
# plus Jan-only and Feb-only impression sums (for the trend feature) and the static
# created date (for page age). All decision-time-safe.
data = con.sql(f"""
    SELECT f.content_hash_id,
           MAX(f.client_hash_id) AS client_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           SUM(CASE WHEN f.report_date >= '2026-01-01' AND f.report_date < '2026-02-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_jan,
           SUM(CASE WHEN f.report_date >= '2026-02-01' AND f.report_date < '2026-03-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_feb,
           c.content_type,
           c.main_intent,
           c.content_created_date,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent, c.content_created_date
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

# --- Core derivations, verbatim from the w05 line (label + rule must reproduce). ---
def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['engagement_rate_fw'] = data['engaged_sessions_fw'] / data['sessions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100,
    include_groups=False
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']
data['below_tier_outcome'] = (data['gap_label'] > 0.1).astype(int)

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')

data['log_impressions_fw'] = np.log1p(data['impressions_fw'])
data['log_sessions_fw'] = np.log1p(data['sessions_fw'])

# --- NEW decision-time-safe features (rule cannot see these). ---
# 1) Impression momentum: Feb impressions / Jan impressions (clip extreme tails).
data['has_jan_imp'] = (data['impressions_jan'] > 0).astype(int)
data['impression_trend'] = (data['impressions_feb'] / data['impressions_jan']).replace(
    [np.inf, -np.inf], np.nan).fillna(0).clip(upper=10)
# 2) Engagement gap: engagement rate below the content-type x intent weighted mean,
#    weighted by sessions (the playbook's eng_target, decision-time-safe).
eng_target = data.groupby(['content_type', 'main_intent'], observed=True).apply(
    lambda g: (g['engagement_rate_fw'] * g['sessions_fw']).sum() / max(g['sessions_fw'].sum(), 1e-9),
    include_groups=False)
eng_target = eng_target.rename('eng_target').reset_index()
data = data.join(eng_target.set_index(['content_type', 'main_intent']),
                 on=['content_type', 'main_intent'], how='left')
data['engagement_gap'] = data['eng_target'] - data['engagement_rate_fw']
data = data.drop(columns=['eng_target'], errors='ignore')

print(f'Class balance: {data["below_tier_outcome"].mean():.1%} positive')
print(f'Pages in data: {len(data):,}  Clients: {data["client_hash_id"].nunique():,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Class balance: 57.5% positive
Pages in data: 120,258  Clients: 47


In [48]:
# --- Leakage diagnostics for the record. None of these columns become features. ---
dim_cols = set(con.sql(f"SELECT * FROM {DIM_CONTENT} LIMIT 0").df().columns)
print('dim_content columns:', sorted(dim_cols))

DECISION = pd.Timestamp('2026-03-01')

meta_cols = [c for c in ['content_updated_date', 'word_count', 'char_count'] if c in dim_cols]
if meta_cols:
    meta = con.sql(f"SELECT content_hash_id, {', '.join(meta_cols)} FROM {DIM_CONTENT}").df()
    data = data.join(meta.set_index('content_hash_id'), on='content_hash_id', how='left')

print()
print('Leakage diagnostics (not used as features):')
if 'content_updated_date' in data.columns:
    upd = pd.to_datetime(data['content_updated_date'], errors='coerce')
    updated_after = (upd > DECISION)
    ra = data.loc[updated_after, 'below_tier_outcome'].mean()
    ro = data.loc[~updated_after, 'below_tier_outcome'].mean()
    print(f'  1. Pages updated AFTER decision: {updated_after.mean():.1%}  '
          f'(below-tier rate {ra:.1%} vs {ro:.1%} for the rest)')
if 'word_count' in data.columns:
    wc = data['word_count'].astype(float)
    if 'content_updated_date' in data.columns:
        print(f'  2. word_count median: {wc[updated_after].median():.0f} on post-decision-updated '
              f'vs {wc[~updated_after].median():.0f} on the rest (a gap = contaminated too)')
    print(f'     share with word_count missing: {(wc == 0).mean():.1%}')
created = pd.to_datetime(data['content_created_date'], errors='coerce')
print(f'  3. pages with created_date AFTER decision: {(created > DECISION).mean():.1%} '
      f'(the few March-created pages are legitimately in the March outcome window)')

# --- Decision-time-safe page-age feature, then drop all raw/diagnostic columns. ---
data['has_created_date'] = created.notna().astype(int)
data['days_since_created'] = (DECISION - created).dt.days.fillna(0).clip(lower=0)

drop_cols = [c for c in ['content_created_date', 'content_updated_date', 'word_count', 'char_count']
             if c in data.columns]
data = data.drop(columns=drop_cols, errors='ignore')
data = data.fillna(0)

print()
print(f'Class balance: {data["below_tier_outcome"].mean():.1%} positive')
print(f'Pages in data: {len(data):,}  Clients: {data["client_hash_id"].nunique():,}')

dim_content columns: ['backlinks', 'category_count', 'char_count', 'client_hash_id', 'competition', 'competition_level', 'content_created_date', 'content_hash_id', 'content_type', 'content_updated_date', 'cpc', 'is_deleted', 'is_published', 'keyword_char_count', 'keyword_created_date', 'keyword_hash_id', 'keyword_token_count', 'last_optimized_date', 'main_intent', 'model_used', 'optimization_eligible_date', 'provider_used', 'search_volume', 'url_char_count', 'url_hash_id', 'word_count']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Leakage diagnostics (not used as features):
  1. Pages updated AFTER decision: 78.2%  (below-tier rate 58.2% vs 55.3% for the rest)
  2. word_count median: 2738 on post-decision-updated vs 3221 on the rest (a gap = contaminated too)
     share with word_count missing: 0.0%
  3. pages with created_date AFTER decision: 7.1% (the few March-created pages are legitimately in the March outcome window)

Class balance: 57.5% positive
Pages in data: 120,258  Clients: 47


## Split design

Four independent client-holdout seeds (42-45), so no single split can fake a
win. Seed 42 is the headline (matches w05: the rule must reproduce 30% / 44% and
the control LR ~50% / 32%); seeds 43-45 are the stability check with fixed
configs. For seed 42 a nested client-holdout inside train (seed 7) picks
hyperparameters and blend weights. The outer holdouts are scored once.

In [49]:
from sklearn.model_selection import GroupShuffleSplit

def client_split(df, seed):
    sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(sp.split(df, groups=df['client_hash_id']))
    return df.iloc[tr].copy(), df.iloc[te].copy()

SEEDS = [42, 43, 44, 45]
splits = {s: client_split(data, s) for s in SEEDS}
train, test = splits[42]

inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
in_train_idx, val_idx = next(inner.split(train, groups=train['client_hash_id']))
in_train = train.iloc[in_train_idx].copy()
val = train.iloc[val_idx].copy()

for s in SEEDS:
    tr, te = splits[s]
    print(f'seed {s}: train {len(tr):,} / {tr["client_hash_id"].nunique()} clients | '
          f'test {len(te):,} / {te["client_hash_id"].nunique()} clients | '
          f'test base rate {te["below_tier_outcome"].mean():.1%}')
print(f'inner (seed 42): in_train {len(in_train):,} | val {len(val):,} | val base rate {val["below_tier_outcome"].mean():.1%}')

seed 42: train 112,968 / 37 clients | test 7,290 / 10 clients | test base rate 15.9%
seed 43: train 92,644 / 37 clients | test 27,614 / 10 clients | test base rate 68.8%
seed 44: train 105,699 / 37 clients | test 14,559 / 10 clients | test base rate 54.4%
seed 45: train 67,365 / 37 clients | test 52,893 / 10 clients | test base rate 66.1%
inner (seed 42): in_train 81,455 | val 31,513 | val base rate 49.5%


## Model zoo

Two feature sets:

- `cur` = the w05 feature set (control; must reproduce w05 numbers).
- `enr` = `cur` plus the three new decision-time-safe features: page age,
  engagement gap, Jan->Feb impression trend (+ presence flags).

Models (same outer holdout, hyperparameters chosen on the inner split only):
the rule (no training), logistic regression on `cur` and `enr`, random forest,
ExtraTrees, HistGradientBoosting, fair XGBoost/LightGBM (logloss, no
scale_pos_weight), CatBoost, a small MLP, kNN, SGD-logistic, GaussianNB, a
rank-blend of the rule with the best tree model, and a stacking meta-model.

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import average_precision_score

num_cur = ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw',
           'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap']
cat_cur = ['content_type', 'main_intent', 'position_tier']
extra_num = ['days_since_created', 'engagement_gap', 'impression_trend', 'has_created_date', 'has_jan_imp']
num_enr = [c for c in num_cur + extra_num if c in data.columns]
cat_enr = list(cat_cur)
print('num_enr:', num_enr)

def make_prep(num, cat):
    return ColumnTransformer([
        ('num', StandardScaler(), num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)])

def build_xy(prep, tr, va, fi, te, num, cat):
    # Preprocessor fits on the FULL train (matches w05 conventions).
    Xtr = prep.fit_transform(tr[num + cat]); ytr = tr['below_tier_outcome']
    Xin = prep.transform(fi[num + cat]); yin = fi['below_tier_outcome']
    Xva = prep.transform(va[num + cat]); yva = va['below_tier_outcome']
    Xte = prep.transform(te[num + cat]); yte = te['below_tier_outcome']
    return Xin, yin, Xva, yva, Xtr, ytr, Xte, yte

def precision_at_k(score, y, k):
    s = score if isinstance(score, pd.Series) else pd.Series(score, index=y.index)
    top = s.nlargest(k).index if len(s) >= k else s.nlargest(len(s)).index
    return y.loc[top].mean()

def rule_score(df):
    has_volume = (df['impressions_fw'] >= 500).astype(int)
    return has_volume * df['tier_ctr_gap'].clip(lower=0) * df['impressions_fw']

def rank_norm(s):
    return pd.Series(s.rank(pct=True), index=s.index)

prep_cur = make_prep(num_cur, cat_cur)
prep_enr = make_prep(num_enr, cat_enr)
Xin_c, yin_c, Xval_c, yval_c, Xtr_c, ytr_c, Xte_c, yte_c = build_xy(
    prep_cur, train, val, in_train, test, num_cur, cat_cur)
Xin_e, yin_e, Xval_e, yval_e, Xtr_e, ytr_e, Xte_e, yte_e = build_xy(
    prep_enr, train, val, in_train, test, num_enr, cat_enr)

bl_val = rule_score(val)
bl_te = rule_score(test)

num_enr: ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw', 'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap', 'days_since_created', 'engagement_gap', 'impression_trend', 'has_created_date', 'has_jan_imp']


In [51]:
# --- Seed-42 zoo. Inner split picks configs; outer holdout is scored once. ---

results = {}
val_prob = {}

def fit_logistic(name, Xin, yin, Xva, yva, Xtr, ytr, Xte, yte, desc):
    m = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
    m.fit(Xin, yin)
    val_prob[name] = pd.Series(m.predict_proba(Xva)[:, 1], index=yva.index)
    m.fit(Xtr, ytr)
    results[name] = (pd.Series(m.predict_proba(Xte)[:, 1], index=yte.index), desc)

def fit_trees(name, model, desc):
    model.fit(Xin_e, yin_e)
    val_prob[name] = pd.Series(model.predict_proba(Xval_e)[:, 1], index=yval_e.index)
    model.fit(Xtr_e, ytr_e)
    results[name] = (pd.Series(model.predict_proba(Xte_e)[:, 1], index=yte_e.index), desc)

def fit_single(name, model, desc):
    model.fit(Xtr_e, ytr_e)
    results[name] = (pd.Series(model.predict_proba(Xte_e)[:, 1], index=yte_e.index), desc)

fit_logistic('lr_cur', Xin_c, yin_c, Xval_c, yval_c, Xtr_c, ytr_c, Xte_c, yte_c,
             'LR on w05 features (control)')
fit_logistic('lr_enr', Xin_e, yin_e, Xval_e, yval_e, Xtr_e, ytr_e, Xte_e, yte_e,
             'LR on safe-enriched features')

fit_trees('rf_enr', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', n_jobs=-1),
          'RandomForest (enr)')
fit_trees('et_enr', ExtraTreesClassifier(n_estimators=300, random_state=42, class_weight='balanced', n_jobs=-1),
          'ExtraTrees (enr)')

# HistGradientBoosting: pick max_depth/learning_rate on the inner split.
hist_configs = [
    {'max_depth': 4, 'learning_rate': 0.05},
    {'max_depth': 6, 'learning_rate': 0.05},
    {'max_depth': 8, 'learning_rate': 0.1},
]
hist_val = {}
for cfg in hist_configs:
    m = HistGradientBoostingClassifier(max_iter=400, class_weight='balanced', random_state=42, **cfg)
    m.fit(Xin_e, yin_e)
    vp = pd.Series(m.predict_proba(Xval_e)[:, 1], index=yval_e.index)
    hist_val[str(cfg)] = precision_at_k(vp, yval_e, 50)
best_cfg = max(hist_val, key=hist_val.get)
print('HistGB chosen config (inner val p@50):', best_cfg, round(hist_val[best_cfg], 3))
cfg = eval(best_cfg)
hist = HistGradientBoostingClassifier(max_iter=400, class_weight='balanced', random_state=42, **cfg)
hist.fit(Xtr_e, ytr_e)
results['histgb_enr'] = (pd.Series(hist.predict_proba(Xte_e)[:, 1], index=yte_e.index), 'HistGB (enr)')
val_prob['histgb_enr'] = pd.Series(hist.predict_proba(Xval_e)[:, 1], index=yval_e.index)

# Fair XGBoost / LightGBM / CatBoost: logloss, early stopping on the inner split,
# no scale_pos_weight (the xp-04 config that degenerated to ~0%).
try:
    import xgboost as xgb
    xm = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=5,
                           subsample=0.8, colsample_bytree=0.8,
                           early_stopping_rounds=50, eval_metric='logloss', random_state=42)
    xm.fit(Xin_e, yin_e, eval_set=[(Xval_e, yval_e)], verbose=False)
    bi = max(xm.best_iteration, 50)
    xm2 = xgb.XGBClassifier(n_estimators=bi, learning_rate=0.03, max_depth=5,
                            subsample=0.8, colsample_bytree=0.8,
                            eval_metric='logloss', random_state=42)
    xm2.fit(Xtr_e, ytr_e)
    results['xgb_enr'] = (pd.Series(xm2.predict_proba(Xte_e)[:, 1], index=yte_e.index), 'XGBoost (enr, logloss)')
    val_prob['xgb_enr'] = pd.Series(xm.predict_proba(Xval_e)[:, 1], index=yval_e.index)
    print(f'XGB best_iteration: {xm.best_iteration}')
except Exception as e:
    import traceback; traceback.print_exc()
    print('XGBoost skipped:', type(e).__name__, '-', e)

try:
    import lightgbm as lgb
    from lightgbm import early_stopping
    lm = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.03, num_leaves=31,
                            subsample=0.8, colsample_bytree=0.8, random_state=42)
    lm.fit(Xin_e, yin_e, eval_set=[(Xval_e, yval_e)], callbacks=[early_stopping(50, verbose=False)])
    bi_l = max(lm.best_iteration_, 50)
    lm2 = lgb.LGBMClassifier(n_estimators=bi_l, learning_rate=0.03, num_leaves=31,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
    lm2.fit(Xtr_e, ytr_e)
    results['lgbm_enr'] = (pd.Series(lm2.predict_proba(Xte_e)[:, 1], index=yte_e.index), 'LightGBM (enr, logloss)')
    val_prob['lgbm_enr'] = pd.Series(lm.predict_proba(Xval_e)[:, 1], index=yval_e.index)
    print(f'LightGBM best_iteration: {lm.best_iteration_}')
except Exception as e:
    import traceback; traceback.print_exc()
    print('LightGBM skipped:', type(e).__name__, '-', e)

try:
    from catboost import CatBoostClassifier
    cb = CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.05, l2_leaf_reg=3.0,
                            early_stopping_rounds=50, random_seed=42, verbose=False)
    cb.fit(Xin_e, yin_e, eval_set=(Xval_e, yval_e))
    bi_c = max(cb.get_best_iteration() or 50, 50)
    cb2 = CatBoostClassifier(iterations=bi_c, depth=6, learning_rate=0.05, l2_leaf_reg=3.0,
                             random_seed=42, verbose=False)
    cb2.fit(Xtr_e, ytr_e)
    results['cat_enr'] = (pd.Series(cb2.predict_proba(Xte_e)[:, 1], index=yte_e.index), 'CatBoost (enr)')
    val_prob['cat_enr'] = pd.Series(cb.predict_proba(Xval_e)[:, 1], index=yval_e.index)
    print(f'CatBoost best_iteration: {bi_c}')
except Exception as e:
    import traceback; traceback.print_exc()
    print('CatBoost skipped:', type(e).__name__, '-', e)

fit_single('mlp_enr', MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, max_iter=500,
                                    early_stopping=True, random_state=42), 'MLP (enr)')
fit_single('knn_enr', KNeighborsClassifier(n_neighbors=100, weights='distance', n_jobs=-1), 'kNN (enr)')
fit_single('sgd_enr', SGDClassifier(loss='log_loss', class_weight='balanced', max_iter=1000, random_state=42),
           'SGD logistic (enr)')
fit_single('nb_enr', GaussianNB(), 'GaussianNB (enr)')

print('methods fitted:', sorted(results))

HistGB chosen config (inner val p@50): {'max_depth': 4, 'learning_rate': 0.05} 1.0
XGB best_iteration: 364
[LightGBM] [Info] Number of positive: 52446, number of negative: 29009
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001658 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2495
[LightGBM] [Info] Number of data points in the train set: 81455, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.643865 -> initscore=0.592178
[LightGBM] [Info] Start training from score 0.592178
[LightGBM] [Info] Number of positive: 68037, number of negative: 44931
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002156 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2536
[Light

/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM best_iteration: 230
CatBoost best_iteration: 227
methods fitted: ['cat_enr', 'et_enr', 'histgb_enr', 'knn_enr', 'lgbm_enr', 'lr_cur', 'lr_enr', 'mlp_enr', 'nb_enr', 'rf_enr', 'sgd_enr', 'xgb_enr']


In [52]:
# --- Rank-blend of the rule with the best tree, and a stacking meta-model.
# Both tuned/fit on the inner split only; the outer holdout is scored once. ---

cands = {k: v for k, v in val_prob.items() if k.startswith(('hist', 'xgb', 'lgbm', 'cat', 'rf', 'et'))}
if cands:
    best_tree = max(cands, key=lambda k: precision_at_k(cands[k], yval_e, 50))
    print('Best tree on inner split:', best_tree)
    rule_rank_val = rank_norm(bl_val)
    model_rank_val = rank_norm(val_prob[best_tree])
    best_w, best_vp = 0.0, 0.0
    for w in np.arange(0.1, 0.95, 0.05):
        mix = w * rule_rank_val + (1 - w) * model_rank_val
        p50 = precision_at_k(mix, yval_e, 50)
        if p50 > best_vp:
            best_vp, best_w = p50, w
    print(f'Blend weight on rule (inner split): w={best_w:.2f}, val p@50={best_vp:.3f}')
    results['blend_rule_%s' % best_tree] = (
        best_w * rank_norm(bl_te) + (1 - best_w) * rank_norm(results[best_tree][0]),
        f'rank-blend rule + {best_tree} (w={best_w:.2f})')

stack_feats = [c for c in ['lr_enr', 'histgb_enr', 'xgb_enr', 'lgbm_enr', 'cat_enr'] if c in val_prob]
if len(stack_feats) >= 2:
    cols = ['rule'] + stack_feats
    V = pd.concat([rank_norm(bl_val)] + [rank_norm(val_prob[c]) for c in stack_feats], axis=1)
    T = pd.concat([rank_norm(bl_te)] + [rank_norm(results[c][0]) for c in stack_feats], axis=1)
    meta = LogisticRegression(random_state=42, max_iter=1000)
    meta.fit(V, yval_e)
    results['stack_meta'] = (pd.Series(meta.predict_proba(T)[:, 1], index=yte_e.index),
                             'stacking meta (rule + %d models)' % len(stack_feats))
    print('stacking meta on:', cols)

results['rule'] = (bl_te, 'transparent rule')

Best tree on inner split: rf_enr
Blend weight on rule (inner split): w=0.10, val p@50=1.000
stacking meta on: ['rule', 'lr_enr', 'histgb_enr', 'xgb_enr', 'lgbm_enr', 'cat_enr']


## Evaluation (seed 42)

One table, every method, same cutoffs, plus AUPRC and lift@50, and the full
precision@k curve. At a 15.9% base rate, precision@10 rests on ~1.6 expected
hits, so the curve and AUPRC matter more than any single point.

In [53]:
ks = list(range(10, 210, 10))
rows = []
curve = {}
auprc = {}
for name, (score, desc) in results.items():
    pre = {k: precision_at_k(score, yte_e, k) for k in ks}
    curve[name] = pre
    auprc[name] = average_precision_score(yte_e, score)
    rows.append({'method': name, 'desc': desc, 'p10': pre[10], 'p50': pre[50], 'p100': pre[100],
                 'auprc': auprc[name], 'lift50': pre[50] / yte_e.mean()})

base = yte_e.mean()
disp = pd.DataFrame(rows).sort_values('p50', ascending=False)
print('Seed 42 client-holdout (base rate %.1f%%):' % (base * 100))
print(disp[['method', 'desc', 'p10', 'p50', 'p100', 'auprc']].to_string(
    index=False, float_format=lambda x: f'{x:.1%}'))
print()
print('Lift@50 over base rate:')
print(disp[['method', 'lift50']].to_string(index=False, float_format=lambda x: f'{x:.2f}x'))
print()
curv = pd.DataFrame(curve).T
print('Precision@k curve (rows = k):')
print(curv.to_string(float_format=lambda x: f'{x:.1%}'))

print()
print(f'Expected hits by chance in top 50: {base * 50:.1f}')
print(f'Rule hits in top 50: {curve["rule"][50] * 50:.1f}')
best = disp.iloc[0]['method']
print(f'Best ({best}) hits in top 50: {curve[best][50] * 50:.1f}')

Seed 42 client-holdout (base rate 15.9%):
           method                              desc    p10    p50   p100  auprc
       histgb_enr                      HistGB (enr) 100.0% 100.0% 100.0%  38.4%
          xgb_enr            XGBoost (enr, logloss) 100.0% 100.0%  99.0%  34.7%
         lgbm_enr           LightGBM (enr, logloss) 100.0% 100.0%  96.0%  34.8%
          cat_enr                    CatBoost (enr) 100.0% 100.0% 100.0%  41.5%
          mlp_enr                         MLP (enr) 100.0%  96.0%  94.0%  37.0%
       stack_meta   stacking meta (rule + 5 models)  90.0%  78.0%  78.0%  34.3%
blend_rule_rf_enr rank-blend rule + rf_enr (w=0.10) 100.0%  58.0%  40.0%  22.4%
          sgd_enr                SGD logistic (enr)  70.0%  56.0%  48.0%  24.0%
           nb_enr                  GaussianNB (enr)  60.0%  52.0%  51.0%  24.9%
             rule                  transparent rule  30.0%  44.0%  49.0%  21.8%
           lr_enr      LR on safe-enriched features  60.0%  42.0%  37.0%  23.1

## Robustness: seeds 43-45

The seed-42 headline uses inner-tuned configs. For stability, this cell runs a
FIXED-config zoo (rule, LR, HistGB, XGBoost, LightGBM, a rule+HistGB blend) on
all four seeds, and counts, per seed, which methods beat the rule at BOTH cuts.
Seeds 43-45 have very different base rates (seed 43 was 68.8% in xp-03/04), so
raw precision is not comparable across seeds — relative lift over each seed's
own base rate is the comparable column.

In [54]:
def fixed_zoo(tr, te):
    prep = make_prep(num_enr, cat_enr)
    Xtr = prep.fit_transform(tr[num_enr + cat_enr]); ytr = tr['below_tier_outcome']
    Xte = prep.transform(te[num_enr + cat_enr]); yte = te['below_tier_outcome']
    out = {}
    def add(name, score, desc):
        out[name] = (pd.Series(score, index=yte.index), desc)
    add('rule', rule_score(te), 'transparent rule')
    m = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
    m.fit(Xtr, ytr); add('lr_enr', m.predict_proba(Xte)[:, 1], 'LR')
    h = HistGradientBoostingClassifier(max_iter=400, max_depth=6, learning_rate=0.05,
                                       class_weight='balanced', random_state=42)
    h.fit(Xtr, ytr); add('histgb_enr', h.predict_proba(Xte)[:, 1], 'HistGB')
    try:
        import xgboost as xgb
        x = xgb.XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=5,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='logloss', random_state=42)
        x.fit(Xtr, ytr); add('xgb_enr', x.predict_proba(Xte)[:, 1], 'XGBoost')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('XGBoost fixed-config skipped:', type(e).__name__, '-', e)
    try:
        import lightgbm as lgb
        g = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=31,
                               subsample=0.8, colsample_bytree=0.8, random_state=42)
        g.fit(Xtr, ytr); add('lgbm_enr', g.predict_proba(Xte)[:, 1], 'LightGBM')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('LightGBM fixed-config skipped:', type(e).__name__, '-', e)
    if 'histgb_enr' in out:
        add('blend_fixed', 0.5 * rank_norm(rule_score(te)) + 0.5 * rank_norm(out['histgb_enr'][0]),
            'rule + HistGB blend (w=0.5)')
    return out, yte

per_seed = {}
for s in SEEDS:
    zoo, yte_s = fixed_zoo(*splits[s])
    base_s = yte_s.mean()
    per_seed[s] = {'base': base_s, 'methods': {
        k: {'p10': precision_at_k(v[0], yte_s, 10), 'p50': precision_at_k(v[0], yte_s, 50)}
        for k, v in zoo.items()}}
    print(f'--- seed {s} (base rate {base_s:.1%}) ---')
    for k, v in zoo.items():
        p10 = precision_at_k(v[0], yte_s, 10); p50 = precision_at_k(v[0], yte_s, 50)
        print(f'  {k:14s} p@10={p10:5.1%}  p@50={p50:5.1%}  lift50={p50 / base_s:5.2f}x')

print()
print('Wins vs rule per seed (beat BOTH cuts, fixed-config comparison):')
for s in SEEDS:
    m = per_seed[s]['methods']
    r10, r50 = m['rule']['p10'], m['rule']['p50']
    winners = sorted(k for k in m if k != 'rule'
                     and m[k]['p10'] > r10 + 0.01 and m[k]['p50'] > r50 + 0.01)
    print(f'  seed {s}: {winners if winners else "none"}')

[LightGBM] [Info] Number of positive: 68037, number of negative: 44931
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002032 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 112968, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.602268 -> initscore=0.414924
[LightGBM] [Info] Start training from score 0.414924


/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- seed 42 (base rate 15.9%) ---
  rule           p@10=30.0%  p@50=44.0%  lift50= 2.77x
  lr_enr         p@10=60.0%  p@50=42.0%  lift50= 2.64x
  histgb_enr     p@10=100.0%  p@50=100.0%  lift50= 6.29x
  xgb_enr        p@10=100.0%  p@50=100.0%  lift50= 6.29x
  lgbm_enr       p@10=100.0%  p@50=100.0%  lift50= 6.29x
  blend_fixed    p@10=100.0%  p@50=74.0%  lift50= 4.65x
[LightGBM] [Info] Number of positive: 50197, number of negative: 42447
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001482 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2524
[LightGBM] [Info] Number of data points in the train set: 92644, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.541827 -> initscore=0.167699
[LightGBM] [Info] Start training from score 0.167699


/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- seed 43 (base rate 68.8%) ---
  rule           p@10=100.0%  p@50=96.0%  lift50= 1.40x
  lr_enr         p@10=100.0%  p@50=100.0%  lift50= 1.45x
  histgb_enr     p@10=100.0%  p@50=100.0%  lift50= 1.45x
  xgb_enr        p@10=100.0%  p@50=100.0%  lift50= 1.45x
  lgbm_enr       p@10=100.0%  p@50=100.0%  lift50= 1.45x
  blend_fixed    p@10=100.0%  p@50=100.0%  lift50= 1.45x
[LightGBM] [Info] Number of positive: 61278, number of negative: 44421
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002053 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2543
[LightGBM] [Info] Number of data points in the train set: 105699, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.579741 -> initscore=0.321709
[LightGBM] [Info] Start training from score 0.321709


/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- seed 44 (base rate 54.4%) ---
  rule           p@10=80.0%  p@50=88.0%  lift50= 1.62x
  lr_enr         p@10=100.0%  p@50=100.0%  lift50= 1.84x
  histgb_enr     p@10=100.0%  p@50=100.0%  lift50= 1.84x
  xgb_enr        p@10=100.0%  p@50=100.0%  lift50= 1.84x
  lgbm_enr       p@10=100.0%  p@50=100.0%  lift50= 1.84x
  blend_fixed    p@10=100.0%  p@50=100.0%  lift50= 1.84x
[LightGBM] [Info] Number of positive: 34221, number of negative: 33144
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2521
[LightGBM] [Info] Number of data points in the train set: 67365, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.507994 -> initscore=0.031978
[LightGBM] [Info] Start training from score 0.031978
--- seed 45 (base rate 66.1%) ---
  rule           p@10=100.0%  p@50=9

/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Verdict (vs the pre-registered criterion)

Win requires beating the rule at BOTH precision@10 and precision@50 on the
seed-42 holdout AND confirming on other seeds. Anything less is a documented
negative (or partial) result, and the paper keeps the honest framing: the
transparent rule generalizes better than every trained model on new clients.

In [55]:
prec10 = {k: v[10] for k, v in curve.items()}
prec50 = {k: v[50] for k, v in curve.items()}
rule10, rule50 = prec10['rule'], prec50['rule']
non_rule = [k for k in curve if k != 'rule']
beat10 = sorted(k for k in non_rule if prec10[k] > rule10 + 0.01)
beat50 = sorted(k for k in non_rule if prec50[k] > rule50 + 0.01)
winners = sorted(set(beat10) & set(beat50))
print(f'Rule (seed 42): p@10={rule10:.1%}  p@50={rule50:.1%}')
print(f'Beat rule at both cuts on seed 42: {winners if winners else "none"}')
if winners:
    print('VERDICT (seed 42): PASS - candidate winner(s) exist')
else:
    print('VERDICT (seed 42): no method beats the rule at BOTH cuts. Honest negative/partial result.')

cross = {}
for s in SEEDS:
    m = per_seed[s]['methods']
    r10, r50 = m['rule']['p10'], m['rule']['p50']
    cross[s] = sorted(k for k in m if k != 'rule'
                      and m[k]['p10'] > r10 + 0.01 and m[k]['p50'] > r50 + 0.01)
print()
print('Wins vs rule by seed (fixed-config comparison):')
for s in SEEDS:
    print(f'  seed {s}: {cross[s] if cross[s] else "none"}')
total = sum(len(v) for v in cross.values())
if winners and total >= 2:
    print('VERDICT: PASS - winner(s) confirmed on multiple seeds')
else:
    print('VERDICT: no method beats the rule at both cuts on the discriminating seed')
    print('  and none is confirmed across seeds. The transparent rule wins the round.')

# --- Receipt for the README log. ---
import json as _json
from datetime import date

def find_repo_root():
    d = Path.cwd()
    for p in [d, *d.parents]:
        if (p / '.git').exists() or (p / 'submission').is_dir():
            return p
    return d

ROOT = find_repo_root()
out_dir = ROOT / 'work' / 'outputs'
out_dir.mkdir(exist_ok=True)
receipt = {
    'run_id': 'xp-05',
    'date': date.today().isoformat(),
    'seed42_base_rate': float(base),
    'seed42_precision': {k: {'p10': v[10], 'p50': v[50], 'p100': v[100]} for k, v in curve.items()},
    'seed42_auprc': {k: float(v) for k, v in auprc.items()},
    'methods': {k: v[1] for k, v in results.items()},
    'wins_vs_rule_by_seed': {str(s): cross[s] for s in SEEDS},
}
out_path = out_dir / 'xp_model_metrics.json'
out_path.write_text(_json.dumps(receipt, indent=2, default=float))
print('Wrote', out_path)

Rule (seed 42): p@10=30.0%  p@50=44.0%
Beat rule at both cuts on seed 42: ['blend_rule_rf_enr', 'cat_enr', 'histgb_enr', 'lgbm_enr', 'mlp_enr', 'nb_enr', 'sgd_enr', 'stack_meta', 'xgb_enr']
VERDICT (seed 42): PASS - candidate winner(s) exist

Wins vs rule by seed (fixed-config comparison):
  seed 42: ['blend_fixed', 'histgb_enr', 'lgbm_enr', 'xgb_enr']
  seed 43: none
  seed 44: ['blend_fixed', 'histgb_enr', 'lgbm_enr', 'lr_enr', 'xgb_enr']
  seed 45: none
VERDICT: PASS - winner(s) confirmed on multiple seeds
Wrote /Users/apple/Desktop/balaji/sem7/flyrank/flyrank-ml-internship/work/outputs/xp_model_metrics.json
